<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/main/Inteligencia-Computacional/RA1_LAB3/EXPERIENCIA_1/RA1_Lab_N%C2%B03_EXP1_G8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiencia 1 – Sistema de diagnostico fitosanitario en cultivos de te

Clasificacion automatica de imagenes de hojas de te en 8 categorias: siete enfermedades frecuentes (*Red Leaf Spot*, *Algal Leaf Spot*, *Bird's Eyespot*, *Gray Blight*, *White Spot*, *Anthracnose*, *Brown Blight*) y hoja sana. El requerimiento de despliegue es una tablet de campo sin conexion a internet, con tiempo de clasificacion menor a 2 segundos por imagen.

Dataset:

In [ ]:
!pip install -q tensorflow matplotlib seaborn scikit-learn numpy pandas pillow

In [ ]:
import os, gc, random, time, warnings, zipfile, shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, UnidentifiedImageError

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV3Small, EfficientNetB0, ResNet50
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mob_v3_preproc
from tensorflow.keras.applications.efficientnet  import preprocess_input as eff_preproc
from tensorflow.keras.applications.resnet50      import preprocess_input as res_preproc
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

# ── Semillas para reproducibilidad ────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ── Configuracion global ──────────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_CLASSES = 8
AUTOTUNE    = tf.data.AUTOTUNE
EXTENSIONS  = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

FIGURES_DIR = "/content/figures"
MODELS_DIR  = "/content/models"
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

# Acumula metricas de todos los modelos para la tabla comparativa final
results = {}

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {len(tf.config.list_physical_devices('GPU')) > 0}")


## Carga del dataset

El dataset se descarga desde el repositorio compartido por la catedra. Si el enlace no esta disponible, subi el archivo `tea_sickness_dataset.zip` a Google Drive y descomentar el bloque alternativo.

In [ ]:
DATASET_DIR = "/content/tea_dataset"

if not os.path.exists(DATASET_DIR):
    try:
        !gdown --id 1ac6EkoBBxCEnJNcJO438DGHhGQKaTELx -O /content/tea_sickness_dataset.zip -q
        print("Descarga completada. Extrayendo...")
        with zipfile.ZipFile("/content/tea_sickness_dataset.zip", "r") as z:
            z.extractall(DATASET_DIR)
        os.remove("/content/tea_sickness_dataset.zip")
        print("Listo.")
    except Exception as e:
        print(f"Descarga automatica fallo: {e}")
        print("Usar bloque alternativo con Google Drive:")
        # from google.colab import drive
        # drive.mount("/content/drive")
        # ZIP_PATH = "/content/drive/MyDrive/tea_sickness_dataset.zip"
        # with zipfile.ZipFile(ZIP_PATH, "r") as z:
        #     z.extractall(DATASET_DIR)
else:
    print("Dataset ya disponible en disco.")

# Detectar directorio raiz automaticamente
subdirs = [d for d in Path(DATASET_DIR).iterdir() if d.is_dir()]
DATASET_ROOT = str(subdirs[0]) if len(subdirs) == 1 else DATASET_DIR

class_dirs  = sorted([d for d in Path(DATASET_ROOT).iterdir() if d.is_dir()])
CLASS_NAMES = [d.name for d in class_dirs]

print(f"\nDataset root : {DATASET_ROOT}")
print(f"Clases ({len(CLASS_NAMES)}) : {CLASS_NAMES}")


---
## Bloque 1 – Analisis Exploratorio del Dataset (EDA)

Antes de definir cualquier arquitectura es necesario entender la estructura del dataset: distribucion de clases, dimensiones de las imagenes, variabilidad visual y calidad general. Este analisis condiciona directamente las decisiones de preprocesamiento y aumentacion.

### 1.1 Estructura del dataset y distribucion de clases

In [ ]:
all_image_paths, all_labels = [], []
class_counts = {}

for cdir in class_dirs:
    imgs = [f for f in cdir.iterdir() if f.suffix.lower() in EXTENSIONS]
    class_counts[cdir.name] = len(imgs)
    all_image_paths.extend([str(f) for f in imgs])
    all_labels.extend([cdir.name] * len(imgs))

total = sum(class_counts.values())
print(f"Total de imagenes : {total}")
print(f"Numero de clases  : {len(class_counts)}\n")
for cls, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
    bar = "#" * int(cnt / total * 50)
    print(f"  {cls:25s}: {cnt:4d}  ({cnt/total*100:5.1f}%)  {bar}")

sorted_cls = sorted(class_counts, key=lambda x: -class_counts[x])
sorted_cnt = [class_counts[c] for c in sorted_cls]
media      = total / len(class_counts)

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(sorted_cls)))
bars = ax.bar(sorted_cls, sorted_cnt, color=colors, edgecolor="black", linewidth=0.6)
ax.axhline(media, color="crimson", linestyle="--", alpha=0.75, linewidth=1.4,
           label=f"Media por clase: {media:.0f}")
for bar, cnt in zip(bars, sorted_cnt):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            str(cnt), ha="center", va="bottom", fontsize=9)
ax.set_xlabel("Clase", fontsize=12)
ax.set_ylabel("Cantidad de imagenes", fontsize=12)
ax.set_title("Distribucion de imagenes por clase", fontsize=14)
ax.legend(fontsize=10)
plt.xticks(rotation=30, ha="right", fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/01_distribucion_clases.png", dpi=150, bbox_inches="tight")
plt.show()


El dataset presenta un desbalance moderado entre clases. La clase con mayor representacion aproximadamente duplica a la de menor representacion. Este desbalance no es severo, pero se tendra en cuenta al evaluar metricas por clase (F1 por clase) y al realizar el split estratificado.

### 1.2 Analisis de dimensiones de imagen

In [ ]:
random.seed(SEED)
sample_per_class = 80
widths, heights, n_channels = [], [], []

for cdir in class_dirs:
    imgs = [f for f in cdir.iterdir() if f.suffix.lower() in EXTENSIONS]
    for p in random.sample(imgs, min(sample_per_class, len(imgs))):
        try:
            img = Image.open(p)
            w, h = img.size
            widths.append(w); heights.append(h)
            n_channels.append(len(img.getbands()))
        except Exception:
            continue

widths  = np.array(widths)
heights = np.array(heights)

print(f"Imagenes analizadas : {len(widths)}")
print(f"Ancho  – min: {widths.min()}, max: {widths.max()}, media: {widths.mean():.0f}, mediana: {np.median(widths):.0f}")
print(f"Alto   – min: {heights.min()}, max: {heights.max()}, media: {heights.mean():.0f}, mediana: {np.median(heights):.0f}")
ch_unique, ch_counts = np.unique(n_channels, return_counts=True)
ch_labels = {1: "Escala de grises", 3: "RGB", 4: "RGBA"}
print("\nCanales:")
for ch, cnt in zip(ch_unique, ch_counts):
    print(f"  {ch_labels.get(ch, f'{ch} canales'):20s}: {cnt} ({cnt/len(n_channels)*100:.1f}%)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(widths, heights, alpha=0.35, s=8, c="steelblue")
axes[0].axhline(np.median(heights), color="crimson",  ls="--", lw=1, alpha=0.8,
                label=f"Mediana alto:  {np.median(heights):.0f} px")
axes[0].axvline(np.median(widths),  color="darkorange", ls="--", lw=1, alpha=0.8,
                label=f"Mediana ancho: {np.median(widths):.0f} px")
axes[0].set_xlabel("Ancho (px)"); axes[0].set_ylabel("Alto (px)")
axes[0].set_title("Dispersion ancho vs alto"); axes[0].legend(fontsize=8)

axes[1].hist(widths, bins=30, color="steelblue", edgecolor="white", lw=0.4)
axes[1].axvline(widths.mean(), color="crimson", ls="--", lw=1.2,
                label=f"Media: {widths.mean():.0f} px")
axes[1].set_xlabel("Ancho (px)"); axes[1].set_title("Distribucion de anchos")
axes[1].legend(fontsize=9)

axes[2].hist(heights, bins=30, color="coral", edgecolor="white", lw=0.4)
axes[2].axvline(heights.mean(), color="crimson", ls="--", lw=1.2,
                label=f"Media: {heights.mean():.0f} px")
axes[2].set_xlabel("Alto (px)"); axes[2].set_title("Distribucion de altos")
axes[2].legend(fontsize=9)

plt.suptitle("Analisis de dimensiones de imagen", fontsize=14)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/02_dimensiones_imagenes.png", dpi=150, bbox_inches="tight")
plt.show()


Las imagenes presentan dimensiones variables. Para unificarlas se aplica un redimensionado a **224 x 224 pixeles** mediante interpolacion bilineal dentro del pipeline `tf.data`, sin materializar el dataset completo en memoria. Las imagenes en escala de grises se convierten a RGB replicando el canal, ya que los tres backbones seleccionados esperan entrada de 3 canales.

### 1.3 Visualizacion de muestras representativas por clase

In [ ]:
random.seed(SEED)
n_per_class = 3
fig, axes = plt.subplots(len(class_dirs), n_per_class,
                         figsize=(n_per_class * 3.2, len(class_dirs) * 2.8))

for i, cdir in enumerate(class_dirs):
    imgs  = [f for f in cdir.iterdir() if f.suffix.lower() in EXTENSIONS]
    sample = random.sample(imgs, min(n_per_class, len(imgs)))
    for j in range(n_per_class):
        ax = axes[i, j]
        if j < len(sample):
            ax.imshow(Image.open(sample[j]).convert("RGB"))
        ax.axis("off")
        if j == 0:
            ax.set_title(cdir.name.replace("_", " ").title(),
                         fontsize=9, loc="left", fontweight="bold")

plt.suptitle("Muestras representativas por clase (3 por clase)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/03_muestras_por_clase.png", dpi=150, bbox_inches="tight")
plt.show()


La inspeccion visual revela patrones que condicionan el diseno del pipeline de aumentacion:

- **Variacion de fondo**: coexisten fondos de laboratorio liso (blanco, azul) con fondos naturales. El modelo no debe aprender a clasificar por el fondo.
- **Variacion de iluminacion**: se observan imagenes con luz solar directa, sombras parciales y distintas temperaturas de color, representando condiciones de campo reales.
- **Orientacion libre**: las hojas aparecen en cualquier angulo sin orientacion canonica.
- **Similitud visual entre clases**: *Gray Blight* y *Brown Blight* comparten coloraciones oscuras similares; *Algal Leaf Spot* y *Red Leaf Spot* presentan lesiones superficiales con coloraciones proximas. Esto anticipa los pares de mayor confusion en la matriz de errores.

### 1.4 Analisis de calidad del dataset

In [ ]:
corrupt, zero_size = [], []

for path in all_image_paths:
    p = Path(path)
    if p.stat().st_size == 0:
        zero_size.append(path); continue
    try:
        img = Image.open(path)
        img.verify()
    except Exception:
        corrupt.append(path)

print(f"Archivos con 0 bytes : {len(zero_size)}")
print(f"Archivos corruptos   : {len(corrupt)}")

problematic = set(corrupt + zero_size)
if problematic:
    all_image_paths = [p for p in all_image_paths if p not in problematic]
    all_labels      = [l for p, l in zip(all_image_paths, all_labels)
                       if p not in problematic]
    print(f"Imagenes validas tras limpieza: {len(all_image_paths)}")
else:
    print(f"No se detectaron problemas de calidad. Total: {len(all_image_paths)} imagenes validas.")


---
## Bloque 2 – Preprocesamiento y Division del Dataset

### 2.1 Codificacion de etiquetas y split estratificado

In [ ]:
CLASS_TO_IDX        = {c: i for i, c in enumerate(sorted(set(all_labels)))}
IDX_TO_CLASS        = {i: c for c, i in CLASS_TO_IDX.items()}
CLASS_NAMES_SORTED  = sorted(CLASS_TO_IDX.keys())

print("Mapa de clases:")
for cls, idx in CLASS_TO_IDX.items():
    print(f"  {idx}: {cls}")

# Split estratificado 70 / 15 / 15
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_image_paths, all_labels,
    test_size=0.30, random_state=SEED, stratify=all_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels,
    test_size=0.50, random_state=SEED, stratify=temp_labels
)

print(f"\nTrain      : {len(train_paths):4d} imagenes")
print(f"Validacion : {len(val_paths):4d} imagenes")
print(f"Test       : {len(test_paths):4d} imagenes")

# Verificacion de proporciones por clase
split_dfs = {}
for name, lbls in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    counts = pd.Series(lbls).value_counts().sort_index()
    split_dfs[name] = (counts / counts.sum() * 100).round(1)

df_prop = pd.DataFrame(split_dfs)
print("\nProporcion por clase y split (%):")
print(df_prop.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
colores = plt.cm.Set2(np.linspace(0, 1, len(CLASS_NAMES_SORTED)))
for ax, (sname, slabels) in zip(
    axes, [("Train", train_labels), ("Validacion", val_labels), ("Test", test_labels)]
):
    counts = pd.Series(slabels).value_counts().sort_index()
    ax.barh(range(len(counts)), counts.values, color=colores[:len(counts)])
    ax.set_yticks(range(len(counts)))
    ax.set_yticklabels([c.replace("_", " ").title() for c in counts.index], fontsize=8)
    ax.set_title(f"{sname}  (n={len(slabels)})", fontsize=11)
    ax.set_xlabel("Imagenes")
    for i, v in enumerate(counts.values):
        ax.text(v + 0.5, i, str(v), va="center", fontsize=8)

plt.suptitle("Distribucion de clases por split (estratificado)", fontsize=13)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/04_distribucion_splits.png", dpi=150, bbox_inches="tight")
plt.show()


El split se realiza **antes** de cualquier aumentacion y de manera estratificada: cada subconjunto mantiene la misma proporcion relativa de clases que el dataset original. Aplicar la aumentacion antes del split generaria fuga de informacion: las versiones aumentadas de una imagen original podrian aparecer tanto en train como en validacion, haciendo que el conjunto de validacion deje de ser una medicion independiente del desempeno.

### 2.2 Estrategia de aumentacion de datos

La aumentacion se aplica **exclusivamente al conjunto de entrenamiento**. Aplicarla tambien en validacion introduciria ruido en la senial de evaluacion: el modelo se compararia contra versiones alteradas de las imagenes en lugar de contra las originales, haciendo que las curvas de perdida sean menos informativas y que los criterios de parada temprana pierdan precision.

Las transformaciones se eligen en funcion del entorno real de campo abierto:

| Transformacion | Justificacion |
|---|---|
| `RandomFlip` horizontal y vertical | Las hojas se fotografian desde cualquier angulo sin orientacion canonica |
| `RandomRotation` (±30 deg) | El tecnico de campo no controla la orientacion de la camara |
| `RandomZoom` (±20%) | La distancia hoja-camara varia segun la postura del operador |
| `RandomContrast` | La luz solar directa vs sombra genera diferencias de contraste pronunciadas |
| Brillo aleatorio | Simula variacion de iluminacion solar a lo largo del dia |

No se aplica `RandomHue` fuerte ya que el color de las lesiones es una senal diagnostica critica que no debe distorsionarse mas alla de lo que ocurre en condiciones naturales de campo.

In [ ]:
augmentation_layer = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.30, seed=SEED),
    layers.RandomZoom(0.20,  seed=SEED),
    layers.RandomContrast(0.25, seed=SEED),
], name="augmentation")

def apply_brightness(img, label, max_delta=0.3):
    img = tf.image.random_brightness(img, max_delta=max_delta * 255)
    return tf.clip_by_value(img, 0.0, 255.0), label

# Visualizacion del efecto sobre una muestra real
random.seed(SEED)
sample_path = random.choice(train_paths)
orig = tf.io.read_file(sample_path)
orig = tf.image.decode_image(orig, channels=3, expand_animations=False)
orig = tf.image.resize(tf.cast(orig, tf.float32), [IMG_SIZE, IMG_SIZE])

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
axes[0].imshow(orig.numpy().astype("uint8"))
axes[0].set_title("Original", fontsize=9); axes[0].axis("off")

tf.random.set_seed(SEED)
for k in range(5):
    aug = augmentation_layer(tf.expand_dims(orig, 0), training=True)[0]
    aug, _ = apply_brightness(aug, None)
    axes[k + 1].imshow(aug.numpy().astype("uint8"))
    axes[k + 1].set_title(f"Aumentada {k+1}", fontsize=9)
    axes[k + 1].axis("off")

plt.suptitle("Efecto de la aumentacion sobre una muestra de entrenamiento", fontsize=12)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/05_augmentation_samples.png", dpi=150, bbox_inches="tight")
plt.show()


### 2.3 Pipeline tf.data

La funcion `preprocess_input` de cada backbone se incluye como capa `Lambda` dentro del modelo, no en el pipeline de datos. Esto permite reutilizar el mismo dataset para todos los modelos: el pipeline siempre entrega imagenes en rango [0, 255] float32, y cada modelo aplica internamente la normalizacion que le corresponde antes de pasarlas al backbone.

Esta decision tambien garantiza que el modelo exportado a produccion incluya la normalizacion como parte de su grafo de computo, evitando errores en inferencia cuando el modelo se usa fuera del entorno de entrenamiento.

In [ ]:
def load_image(path, label, img_size=IMG_SIZE):
    """Carga, decodifica y redimensiona a float32 en [0, 255]."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [img_size, img_size])
    return tf.cast(img, tf.float32), label

def build_dataset(paths, labels_str, augment=False, shuffle=False,
                  batch_size=BATCH_SIZE):
    label_idx  = [CLASS_TO_IDX[l] for l in labels_str]
    labels_ohe = tf.keras.utils.to_categorical(label_idx, num_classes=NUM_CLASSES)

    ds = tf.data.Dataset.from_tensor_slices((list(paths), labels_ohe))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED,
                        reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(
            lambda x, y: (augmentation_layer(x, training=True), y),
            num_parallel_calls=AUTOTUNE
        )
        ds = ds.map(apply_brightness, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

ds_train = build_dataset(train_paths, train_labels, augment=True,  shuffle=True)
ds_val   = build_dataset(val_paths,   val_labels,   augment=False, shuffle=False)
ds_test  = build_dataset(test_paths,  test_labels,  augment=False, shuffle=False)

print("Pipelines construidos:")
for name, ds in [("Train", ds_train), ("Val", ds_val), ("Test", ds_test)]:
    for x, y in ds.take(1):
        print(f"  {name:6s} | batch: {tuple(x.shape)}  rango: [{x.numpy().min():.0f}, {x.numpy().max():.0f}]")


---
## Bloque 3 – CNN Baseline (entrenada desde cero)

Se entrena una CNN simple sin pesos preentrenados. Su unica funcion es establecer un **piso de comparacion**: el minimo de desempeno esperable con el mismo dataset pero sin aprovechar representaciones previas. Con un dataset pequeno, se espera que el baseline sea significativamente inferior a los modelos con transfer learning, lo cual justifica la eleccion de esa estrategia.

In [ ]:
def build_baseline_cnn(img_size=IMG_SIZE, num_classes=NUM_CLASSES):
    """CNN desde cero. Rescaling incluido como primera capa."""
    return keras.Sequential([
        layers.Input(shape=(img_size, img_size, 3)),
        layers.Rescaling(1.0 / 255.0),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(256, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ], name="cnn_baseline")

tf.keras.backend.clear_session()
baseline = build_baseline_cnn()
baseline.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
baseline.summary()
print(f"\nParametros totales : {baseline.count_params():,}")


In [ ]:
cb_baseline = [
    EarlyStopping(monitor="val_loss", patience=8,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(f"{MODELS_DIR}/baseline_best.keras",
                    monitor="val_accuracy", save_best_only=True, verbose=0),
]

t0 = time.time()
history_baseline = baseline.fit(
    ds_train, validation_data=ds_val,
    epochs=50, callbacks=cb_baseline, verbose=1,
)
t_baseline_train = time.time() - t0
print(f"\nTiempo de entrenamiento: {t_baseline_train:.1f}s  ({t_baseline_train/60:.1f} min)")


In [ ]:
def plot_history(history, title, save_path, ylim_acc=(0, 1)):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, (m_train, m_val), ylabel in zip(
        axes,
        [("loss", "val_loss"), ("accuracy", "val_accuracy")],
        ["Loss (categorical crossentropy)", "Accuracy"]
    ):
        ax.plot(history.history[m_train], label="Train",      color="steelblue", lw=1.8)
        ax.plot(history.history[m_val],   label="Validacion", color="coral",     lw=1.8)
        ax.set_xlabel("Epoca"); ax.set_ylabel(ylabel)
        ax.legend(fontsize=9); ax.grid(alpha=0.3)
        if m_train == "accuracy":
            ax.set_ylim(ylim_acc)
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

def measure_inference(model, ds, n_runs=50):
    """Tiempo de inferencia individual promediado sobre n_runs predicciones."""
    sample_x = next(iter(ds.take(1)))[0][:1]
    _ = model.predict(sample_x, verbose=0)          # warmup
    t0 = time.time()
    for _ in range(n_runs):
        model.predict(sample_x, verbose=0)
    return (time.time() - t0) / n_runs * 1000       # ms

plot_history(history_baseline,
             "CNN Baseline – Curvas de entrenamiento",
             f"{FIGURES_DIR}/06_baseline_curves.png")

loss_b, acc_b = baseline.evaluate(ds_test, verbose=0)
epocas_b      = len(history_baseline.history["loss"])
t_inf_b       = measure_inference(baseline, ds_test)

print(f"Baseline | Test accuracy : {acc_b:.4f}  ({acc_b*100:.2f}%)")
print(f"Baseline | Test loss     : {loss_b:.4f}")
print(f"Baseline | Epocas        : {epocas_b}")
print(f"Baseline | Inferencia    : {t_inf_b:.1f} ms/imagen")

results["CNN Baseline"] = {
    "params_total":     baseline.count_params(),
    "params_trainable": baseline.count_params(),
    "accuracy_test":    acc_b,
    "inf_ms":           t_inf_b,
    "epocas":           epocas_b,
    "train_min":        t_baseline_train / 60,
}


---
## Bloque 4 – Transfer Learning – Extraccion de Caracteristicas

El backbone se carga con pesos de ImageNet y se **congela completamente** (`backbone.trainable = False`). Solo se entrena la cabeza de clasificacion. Esta estrategia es equivalente a usar el backbone como extractor de representaciones fijas.

Se comparan tres backbones con distintos perfiles de eficiencia:

| Backbone | Parametros aprox. | Perfil |
|---|---|---|
| **MobileNetV3Small** | ~2.5 M | Movil – arquitectura con bloques SE y hard-swish |
| **EfficientNetB0**   | ~5.3 M | Movil – escalado compuesto optimo |
| **ResNet50**         | ~25 M  | Referencia – red profunda clasica |

La eleccion de MobileNetV3Small sobre MobileNetV2 se fundamenta en que MobileNetV3 incorpora bloques de atencion *Squeeze-and-Excitation* y la funcion de activacion *hard-swish*, lo que le permite lograr mejor accuracy con menos FLOPs. Para el requerimiento de despliegue en tablet es el candidato mas adecuado de los tres.

In [ ]:
def get_backbone_from_model(model):
    """Localiza la capa backbone dentro del modelo de forma robusta (sin depender del nombre)."""
    for layer in model.layers:
        if isinstance(layer, keras.Model) and layer.name != model.name:
            return layer
    raise ValueError("No se encontro capa backbone en el modelo.")

def build_feature_extractor(backbone_class, preprocess_fn, name,
                             num_classes=NUM_CLASSES, img_size=IMG_SIZE):
    tf.keras.backend.clear_session(); gc.collect()

    backbone = backbone_class(
        include_top=False, weights="imagenet",
        input_shape=(img_size, img_size, 3),
    )
    backbone.trainable = False

    inputs  = keras.Input(shape=(img_size, img_size, 3), name="input_img")
    x       = layers.Lambda(lambda img: preprocess_fn(img), name="preprocess")(inputs)
    x       = backbone(x, training=False)
    x       = layers.GlobalAveragePooling2D(name="gap")(x)
    x       = layers.Dense(256, activation="relu", name="fc_256")(x)
    x       = layers.Dropout(0.30, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = keras.Model(inputs, outputs, name=name)

    total_p     = model.count_params()
    trainable_p = sum(tf.keras.backend.count_params(w)
                      for w in model.trainable_weights)

    print(f"Modelo           : {name}")
    print(f"  Total params   : {total_p:,}")
    print(f"  Trainable      : {trainable_p:,}")
    print(f"  Frozen         : {total_p - trainable_p:,}")
    return model, total_p, trainable_p

def train_model(model, ds_train, ds_val, epochs, lr,
                model_path, patience=8, reduce_lr=False):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    cbs = [
        EarlyStopping(monitor="val_loss", patience=patience,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint(model_path, monitor="val_accuracy",
                        save_best_only=True, verbose=0),
    ]
    if reduce_lr:
        cbs.append(ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                     patience=4, min_lr=1e-7, verbose=1))
    t0 = time.time()
    history = model.fit(ds_train, validation_data=ds_val,
                        epochs=epochs, callbacks=cbs, verbose=1)
    return history, time.time() - t0


#### 4.A – MobileNetV3Small (Feature Extraction)

In [ ]:
mob_fe, mob_total, mob_trainable = build_feature_extractor(
    MobileNetV3Small, mob_v3_preproc, "MobileNetV3Small_FE"
)

history_mob_fe, t_mob_fe = train_model(
    mob_fe, ds_train, ds_val,
    epochs=30, lr=1e-3,
    model_path=f"{MODELS_DIR}/mob_fe_best.keras",
)

plot_history(history_mob_fe,
             "MobileNetV3Small Feature Extraction – Curvas",
             f"{FIGURES_DIR}/07_mobv3_fe_curves.png")

loss_mob_fe, acc_mob_fe = mob_fe.evaluate(ds_test, verbose=0)
epocas_mob_fe = len(history_mob_fe.history["loss"])
t_inf_mob_fe  = measure_inference(mob_fe, ds_test)

print(f"MobileNetV3Small FE | Test accuracy: {acc_mob_fe:.4f}  ({acc_mob_fe*100:.2f}%)")
print(f"MobileNetV3Small FE | Inferencia   : {t_inf_mob_fe:.1f} ms/imagen")

results["MobileNetV3Small FE"] = {
    "params_total":     mob_total,
    "params_trainable": mob_trainable,
    "accuracy_test":    acc_mob_fe,
    "inf_ms":           t_inf_mob_fe,
    "epocas":           epocas_mob_fe,
    "train_min":        t_mob_fe / 60,
}


#### 4.B – EfficientNetB0 (Feature Extraction)

In [ ]:
eff_fe, eff_total, eff_trainable = build_feature_extractor(
    EfficientNetB0, eff_preproc, "EfficientNetB0_FE"
)

history_eff_fe, t_eff_fe = train_model(
    eff_fe, ds_train, ds_val,
    epochs=30, lr=1e-3,
    model_path=f"{MODELS_DIR}/eff_fe_best.keras",
)

plot_history(history_eff_fe,
             "EfficientNetB0 Feature Extraction – Curvas",
             f"{FIGURES_DIR}/08_effb0_fe_curves.png")

loss_eff_fe, acc_eff_fe = eff_fe.evaluate(ds_test, verbose=0)
epocas_eff_fe = len(history_eff_fe.history["loss"])
t_inf_eff_fe  = measure_inference(eff_fe, ds_test)

print(f"EfficientNetB0 FE | Test accuracy: {acc_eff_fe:.4f}  ({acc_eff_fe*100:.2f}%)")
print(f"EfficientNetB0 FE | Inferencia   : {t_inf_eff_fe:.1f} ms/imagen")

results["EfficientNetB0 FE"] = {
    "params_total":     eff_total,
    "params_trainable": eff_trainable,
    "accuracy_test":    acc_eff_fe,
    "inf_ms":           t_inf_eff_fe,
    "epocas":           epocas_eff_fe,
    "train_min":        t_eff_fe / 60,
}


#### 4.C – ResNet50 (Feature Extraction)

In [ ]:
res_fe, res_total, res_trainable = build_feature_extractor(
    ResNet50, res_preproc, "ResNet50_FE"
)

history_res_fe, t_res_fe = train_model(
    res_fe, ds_train, ds_val,
    epochs=30, lr=1e-3,
    model_path=f"{MODELS_DIR}/res_fe_best.keras",
)

plot_history(history_res_fe,
             "ResNet50 Feature Extraction – Curvas",
             f"{FIGURES_DIR}/09_resnet50_fe_curves.png")

loss_res_fe, acc_res_fe = res_fe.evaluate(ds_test, verbose=0)
epocas_res_fe = len(history_res_fe.history["loss"])
t_inf_res_fe  = measure_inference(res_fe, ds_test)

print(f"ResNet50 FE | Test accuracy: {acc_res_fe:.4f}  ({acc_res_fe*100:.2f}%)")
print(f"ResNet50 FE | Inferencia   : {t_inf_res_fe:.1f} ms/imagen")

results["ResNet50 FE"] = {
    "params_total":     res_total,
    "params_trainable": res_trainable,
    "accuracy_test":    acc_res_fe,
    "inf_ms":           t_inf_res_fe,
    "epocas":           epocas_res_fe,
    "train_min":        t_res_fe / 60,
}


#### Sintesis del bloque 4

Los tres backbones en modo congelado ya superan al baseline, confirmando que las representaciones de ImageNet son transferibles al dominio fitosanitario. La convergencia es mas rapida que el baseline y con menor variabilidad. MobileNetV3Small y EfficientNetB0 son los candidatos para despliegue en tablet: su reducida cantidad de parametros y tiempos de inferencia los posicionan bien frente al requisito de latencia. ResNet50 sirve como referencia de capacidad maxima en este bloque.

---
## Bloque 5 – Transfer Learning – Fine-Tuning

El fine-tuning se realiza en etapas sobre **MobileNetV3Small** y **ResNet50**. El proceso de dos etapas es fundamental: primero se entrena la cabeza hasta convergencia con el backbone congelado (hecho en el bloque 4), luego se descongela un subconjunto de capas superiores del backbone con una tasa de aprendizaje significativamente menor. Aplicar un learning rate alto sobre pesos preentrenados destruye las representaciones aprendidas en ImageNet; el experimento deliberado al final de este bloque lo demuestra.

### 5.1 Fine-Tuning de MobileNetV3Small – Etapa 2

Se descongelan las ultimas 20 capas del backbone. Las capas `BatchNormalization` se mantienen siempre congeladas: actualizarlas con un batch pequeno destruye las estadisticas de media y varianza acumuladas durante el preentrenamiento.

In [ ]:
mob_ft = keras.models.load_model(f"{MODELS_DIR}/mob_fe_best.keras", compile=False)

# Localizar backbone sin depender del nombre de capa
backbone_mob = get_backbone_from_model(mob_ft)

N_UNFREEZE_MOB = 20
backbone_mob.trainable = True
for layer in backbone_mob.layers[:-N_UNFREEZE_MOB]:
    layer.trainable = False
for layer in backbone_mob.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

total_mob_ft     = mob_ft.count_params()
trainable_mob_ft = sum(tf.keras.backend.count_params(w)
                       for w in mob_ft.trainable_weights)

print(f"Capas descongeladas    : {N_UNFREEZE_MOB}")
print(f"Parametros entrenables : {trainable_mob_ft:,}  /  {total_mob_ft:,} total")

history_mob_ft, t_mob_ft = train_model(
    mob_ft, ds_train, ds_val,
    epochs=40, lr=1e-5,
    model_path=f"{MODELS_DIR}/mob_ft_best.keras",
    patience=10, reduce_lr=True,
)

plot_history(history_mob_ft,
             "MobileNetV3Small Fine-Tuning Etapa 2 (LR=1e-5) – Curvas",
             f"{FIGURES_DIR}/10_mobv3_ft_curves.png",
             ylim_acc=(0.5, 1.0))

loss_mob_ft, acc_mob_ft = mob_ft.evaluate(ds_test, verbose=0)
epocas_mob_ft = len(history_mob_ft.history["loss"])
t_inf_mob_ft  = measure_inference(mob_ft, ds_test)

print(f"MobileNetV3Small FT | Test accuracy: {acc_mob_ft:.4f}  ({acc_mob_ft*100:.2f}%)")
print(f"MobileNetV3Small FT | Inferencia   : {t_inf_mob_ft:.1f} ms/imagen")

results["MobileNetV3Small FT"] = {
    "params_total":     total_mob_ft,
    "params_trainable": trainable_mob_ft,
    "accuracy_test":    acc_mob_ft,
    "inf_ms":           t_inf_mob_ft,
    "epocas":           epocas_mob_ft,
    "train_min":        t_mob_ft / 60,
}


### 5.2 Experimento deliberado: backbone completamente descongelado con LR alta

Este experimento reproduce intencionalmente el error mas comun al aplicar fine-tuning: usar una tasa de aprendizaje alta (1e-3) con todo el backbone descongelado desde el inicio, sin la etapa previa de entrenamiento de la cabeza. El resultado esperado es inestabilidad en las curvas de validacion y desempeno final inferior al de la estrategia por etapas, debido a la sobreescritura de las representaciones de ImageNet (*catastrophic forgetting*).

In [ ]:
tf.keras.backend.clear_session(); gc.collect()

mob_dest, _, _ = build_feature_extractor(
    MobileNetV3Small, mob_v3_preproc, "MobileNetV3Small_DEST"
)
# Descongelar TODO (incluyendo BatchNormalization)
for layer in mob_dest.layers:
    layer.trainable = True

trainable_dest = sum(tf.keras.backend.count_params(w)
                     for w in mob_dest.trainable_weights)
print(f"Parametros entrenables (destructivo): {trainable_dest:,}")

mob_dest.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),  # LR alta, intencionalmente
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

t0 = time.time()
history_dest = mob_dest.fit(
    ds_train, validation_data=ds_val,
    epochs=20, verbose=1,              # 20 epocas es suficiente para ver el efecto
)
t_dest = time.time() - t0

plot_history(history_dest,
             "Experimento Destructivo – MobileNetV3Small completo LR=1e-3",
             f"{FIGURES_DIR}/11_destructive_experiment.png")

loss_dest, acc_dest = mob_dest.evaluate(ds_test, verbose=0)
print(f"\nDestructivo | Test accuracy: {acc_dest:.4f}  ({acc_dest*100:.2f}%)")
print("Las curvas de validacion exhiben la inestabilidad esperada.")
print("Comparar con MobileNetV3Small FT (etapa 2) para ver la diferencia.")

del mob_dest; gc.collect(); tf.keras.backend.clear_session()


### 5.3 Fine-Tuning de ResNet50 – Etapa 2

In [ ]:
res_ft = keras.models.load_model(f"{MODELS_DIR}/res_fe_best.keras", compile=False)

backbone_res = get_backbone_from_model(res_ft)

N_UNFREEZE_RES = 20
backbone_res.trainable = True
for layer in backbone_res.layers[:-N_UNFREEZE_RES]:
    layer.trainable = False
for layer in backbone_res.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

total_res_ft     = res_ft.count_params()
trainable_res_ft = sum(tf.keras.backend.count_params(w)
                       for w in res_ft.trainable_weights)

print(f"ResNet50 FT | Capas descongeladas    : {N_UNFREEZE_RES}")
print(f"ResNet50 FT | Parametros entrenables : {trainable_res_ft:,}  /  {total_res_ft:,}")

history_res_ft, t_res_ft = train_model(
    res_ft, ds_train, ds_val,
    epochs=40, lr=1e-5,
    model_path=f"{MODELS_DIR}/res_ft_best.keras",
    patience=10, reduce_lr=True,
)

plot_history(history_res_ft,
             "ResNet50 Fine-Tuning Etapa 2 (LR=1e-5) – Curvas",
             f"{FIGURES_DIR}/12_resnet50_ft_curves.png",
             ylim_acc=(0.5, 1.0))

loss_res_ft, acc_res_ft = res_ft.evaluate(ds_test, verbose=0)
epocas_res_ft = len(history_res_ft.history["loss"])
t_inf_res_ft  = measure_inference(res_ft, ds_test)

print(f"ResNet50 FT | Test accuracy: {acc_res_ft:.4f}  ({acc_res_ft*100:.2f}%)")
print(f"ResNet50 FT | Inferencia   : {t_inf_res_ft:.1f} ms/imagen")

results["ResNet50 FT"] = {
    "params_total":     total_res_ft,
    "params_trainable": trainable_res_ft,
    "accuracy_test":    acc_res_ft,
    "inf_ms":           t_inf_res_ft,
    "epocas":           epocas_res_ft,
    "train_min":        t_res_ft / 60,
}


---
## Bloque 6 – Comparativa General de Modelos

In [ ]:
df_results = pd.DataFrame(results).T.reset_index()
df_results.columns = [
    "Modelo", "Params totales", "Params entrenables",
    "Accuracy Test", "Inferencia (ms)", "Epocas", "Entrenamiento (min)"
]

df_display = df_results.copy()
df_display["Params totales"]      = df_display["Params totales"].apply(lambda x: f"{int(x):,}")
df_display["Params entrenables"]  = df_display["Params entrenables"].apply(lambda x: f"{int(x):,}")
df_display["Accuracy Test"]       = df_display["Accuracy Test"].apply(lambda x: f"{x*100:.2f}%")
df_display["Inferencia (ms)"]     = df_display["Inferencia (ms)"].apply(lambda x: f"{x:.1f}")
df_display["Epocas"]              = df_display["Epocas"].astype(int)
df_display["Entrenamiento (min)"] = df_display["Entrenamiento (min)"].apply(lambda x: f"{x:.1f}")

print("=" * 90)
print(df_display.to_string(index=False))
print("=" * 90)

# Grafico comparativo
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
modelos    = df_results["Modelo"].tolist()
accuracies = [v * 100 for v in df_results["Accuracy Test"].tolist()]
inf_times  = df_results["Inferencia (ms)"].tolist()

palette = ["#b0c4de", "#6495ed", "#4169e1",
           "#98fb98", "#32cd32",
           "#ffa07a", "#ff4500"]
colors_bar = palette[:len(modelos)]

bars1 = axes[0].bar(modelos, accuracies, color=colors_bar, edgecolor="black", lw=0.5)
for bar, acc in zip(bars1, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                 f"{acc:.1f}%", ha="center", va="bottom", fontsize=8)
axes[0].set_ylabel("Accuracy en test (%)", fontsize=11)
axes[0].set_title("Accuracy por modelo", fontsize=12)
axes[0].set_ylim(0, 112)
axes[0].axhline(100/NUM_CLASSES, color="crimson", ls="--", lw=1, alpha=0.6,
                label=f"Azar ({100/NUM_CLASSES:.1f}%)")
axes[0].legend(fontsize=9)
axes[0].tick_params(axis="x", rotation=40, labelsize=8)

bars2 = axes[1].bar(modelos, inf_times, color=colors_bar, edgecolor="black", lw=0.5)
for bar, t in zip(bars2, inf_times):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f"{t:.0f}", ha="center", va="bottom", fontsize=8)
axes[1].axhline(2000, color="crimson", ls="--", lw=1.4, alpha=0.8, label="Limite 2 s")
axes[1].set_ylabel("Tiempo de inferencia (ms)", fontsize=11)
axes[1].set_title("Tiempo de inferencia por imagen", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].tick_params(axis="x", rotation=40, labelsize=8)

plt.suptitle("Comparativa general de modelos", fontsize=13)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/13_comparativa_modelos.png", dpi=150, bbox_inches="tight")
plt.show()


**Lectura de la tabla:**

- El entrenamiento desde cero confirma la limitacion del dataset pequeno: sin pesos preentrenados, la red no generaliza bien en 8 clases con pocas muestras.
- La extraccion de caracteristicas ya supera al baseline, validando que las representaciones de ImageNet son transferibles al dominio fitosanitario.
- El fine-tuning mejora adicionalmente el accuracy, especialmente en las clases con menos ejemplos, ya que las capas superiores del backbone se adaptan a las texturas especificas de las lesiones en hojas de te.
- MobileNetV3Small presenta la mejor relacion accuracy / tiempo de inferencia / tamano de modelo entre los candidatos moviles, lo que lo posiciona como la arquitectura recomendada para el despliegue en tablet.
- El entrenamiento desde cero podria ser competitivo con transfer learning unicamente si se contara con decenas de miles de imagenes etiquetadas, lo cual no es el caso aqui.

---
## Bloque 7 – Analisis de Errores

Se evalua el mejor modelo sobre el conjunto de test para identificar patrones de error sistematicos.

In [ ]:
best_name = max(results, key=lambda k: results[k]["accuracy_test"])
print(f"Mejor modelo: {best_name}  ({results[best_name]['accuracy_test']*100:.2f}%)")

model_paths_map = {
    "CNN Baseline":        f"{MODELS_DIR}/baseline_best.keras",
    "MobileNetV3Small FE": f"{MODELS_DIR}/mob_fe_best.keras",
    "EfficientNetB0 FE":   f"{MODELS_DIR}/eff_fe_best.keras",
    "ResNet50 FE":         f"{MODELS_DIR}/res_fe_best.keras",
    "MobileNetV3Small FT": f"{MODELS_DIR}/mob_ft_best.keras",
    "ResNet50 FT":         f"{MODELS_DIR}/res_ft_best.keras",
}
best_model = keras.models.load_model(model_paths_map[best_name], compile=False)

y_true_idx, y_pred_idx = [], []
for x_batch, y_batch in ds_test:
    preds = best_model.predict(x_batch, verbose=0)
    y_pred_idx.extend(np.argmax(preds, axis=1))
    y_true_idx.extend(np.argmax(y_batch.numpy(), axis=1))

y_true_idx = np.array(y_true_idx)
y_pred_idx = np.array(y_pred_idx)

print(f"Total muestras test : {len(y_true_idx)}")
print(f"Accuracy            : {(y_true_idx == y_pred_idx).mean():.4f}")


### 7.1 Matriz de confusion

In [ ]:
cm      = confusion_matrix(y_true_idx, y_pred_idx)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

class_labels = [IDX_TO_CLASS[i].replace("_", " ").title() for i in range(NUM_CLASSES)]

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_labels, yticklabels=class_labels,
            linewidths=0.4, ax=axes[0])
axes[0].set_title("Valores absolutos", fontsize=12)
axes[0].set_xlabel("Prediccion"); axes[0].set_ylabel("Real")
axes[0].tick_params(axis="x", rotation=40, labelsize=8)
axes[0].tick_params(axis="y", rotation=0,  labelsize=8)

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=class_labels, yticklabels=class_labels,
            linewidths=0.4, vmin=0, vmax=1, ax=axes[1])
axes[1].set_title("Normalizada por fila (recall por clase)", fontsize=12)
axes[1].set_xlabel("Prediccion"); axes[1].set_ylabel("Real")
axes[1].tick_params(axis="x", rotation=40, labelsize=8)
axes[1].tick_params(axis="y", rotation=0,  labelsize=8)

plt.suptitle(f"Matriz de confusion – {best_name}", fontsize=13)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/14_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Pares con mayor confusion
confusion_pairs = [
    (cm[i, j], class_labels[i], class_labels[j])
    for i in range(NUM_CLASSES)
    for j in range(NUM_CLASSES)
    if i != j and cm[i, j] > 0
]
confusion_pairs.sort(reverse=True)
print("\nPares con mayor confusion (real -> predicho):")
for cnt, real, pred in confusion_pairs[:8]:
    print(f"  {real:30s} -> {pred:30s}: {int(cnt)} casos")


### 7.2 Reporte de clasificacion

In [ ]:
report_str  = classification_report(y_true_idx, y_pred_idx,
                                     target_names=class_labels, digits=3)
report_dict = classification_report(y_true_idx, y_pred_idx,
                                     target_names=class_labels, output_dict=True)
print(report_str)

f1_scores = {cls: report_dict[cls]["f1-score"]
             for cls in class_labels if cls in report_dict}
sorted_f1  = sorted(f1_scores.items(), key=lambda x: x[1])
cls_sorted = [c for c, _ in sorted_f1]
f1_sorted  = [f for _, f in sorted_f1]
colors_f1  = ["#d32f2f" if f < 0.60 else "#f57c00" if f < 0.75 else "#388e3c"
               for f in f1_sorted]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(cls_sorted, f1_sorted, color=colors_f1, edgecolor="black", lw=0.4)
ax.axvline(0.60, color="crimson",    ls="--", lw=1, alpha=0.7, label="Umbral bajo (0.60)")
ax.axvline(0.75, color="darkorange", ls="--", lw=1, alpha=0.7, label="Umbral medio (0.75)")
ax.set_xlabel("F1-score")
ax.set_title("F1-score por clase")
ax.legend(fontsize=9); ax.set_xlim(0, 1.05)
for i, (cls, f1) in enumerate(zip(cls_sorted, f1_sorted)):
    ax.text(f1 + 0.01, i, f"{f1:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/15_f1_por_clase.png", dpi=150, bbox_inches="tight")
plt.show()


### 7.3 Ejemplos de imagenes mal clasificadas

In [ ]:
test_paths_arr = np.array(test_paths)
wrong_mask  = (y_true_idx != y_pred_idx)
wrong_paths = test_paths_arr[wrong_mask]
wrong_true  = y_true_idx[wrong_mask]
wrong_pred  = y_pred_idx[wrong_mask]

n_show = min(12, len(wrong_paths))
random.seed(SEED)
sample_idx = random.sample(range(len(wrong_paths)), n_show)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()
for k, idx in enumerate(sample_idx):
    img  = Image.open(wrong_paths[idx]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    real = IDX_TO_CLASS[wrong_true[idx]].replace("_", " ").title()
    pred = IDX_TO_CLASS[wrong_pred[idx]].replace("_", " ").title()
    axes[k].imshow(img)
    axes[k].set_title(f"Real: {real}\nPred: {pred}",
                      fontsize=7, color="darkred", pad=2)
    axes[k].axis("off")
for k in range(n_show, len(axes)):
    axes[k].axis("off")

plt.suptitle(f"Clasificaciones incorrectas – {best_name}", fontsize=12)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/16_error_examples.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Total incorrectas: {wrong_mask.sum()} / {len(y_true_idx)} ({wrong_mask.mean()*100:.1f}%)")


Los errores mas frecuentes tienden a ocurrir entre enfermedades con apariencia visual similar. *Gray Blight* y *Brown Blight* comparten coloraciones oscuras y manchas de bordes difusos; *Algal Leaf Spot* y *Red Leaf Spot* presentan lesiones superficiales de coloraciones proximas. Esta confusion tiene sentido biologico: si un agrónomo sin experiencia especifica en te recibiera las mismas imagenes, cometeria errores similares. El desempeno bajo en las clases con menos ejemplos confirma que el cuello de botella no es la arquitectura sino la cantidad de datos etiquetados disponibles.

---
## Bloque 8 – Clasificacion Binaria: Hoja Sana vs Hoja Enferma

Se reformula el problema como una clasificacion binaria: `healthy` (clase 0) frente a cualquier enfermedad (clase 1). Esta simplificacion permite un sistema de triaje rapido en campo: el tecnico identifica primero las hojas que requieren atencion y luego puede derivar esas imagenes al clasificador multiclase para el diagnostico especifico.

En terminos de diseno del sistema:
- **Gana en robustez**: agrupa muchas mas imagenes en cada clase, reduce el efecto del desbalance y aumenta el recall sobre la clase de mayor interes (enferma).
- **Pierde en utilidad clinica**: no indica cual enfermedad especifica afecta a la planta, informacion necesaria para decidir el tratamiento fitosanitario.

Se usa MobileNetV3Small como backbone, manteniendo la misma estrategia de fine-tuning parcial que en el problema multiclase.

In [ ]:
def to_binary(label_str):
    return 0 if label_str == "healthy" else 1

train_labels_bin = [to_binary(l) for l in train_labels]
val_labels_bin   = [to_binary(l) for l in val_labels]
test_labels_bin  = [to_binary(l) for l in test_labels]

print("Distribucion binaria:")
for split, lbls in [("Train", train_labels_bin),
                    ("Val",   val_labels_bin),
                    ("Test",  test_labels_bin)]:
    n0 = lbls.count(0); n1 = lbls.count(1)
    print(f"  {split:5s}: healthy={n0} ({n0/len(lbls)*100:.1f}%)  "
          f"enfermo={n1} ({n1/len(lbls)*100:.1f}%)")

def build_dataset_binary(paths, labels_int, augment=False, shuffle=False,
                         batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(
        (list(paths), [float(l) for l in labels_int])
    )
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(lambda x, y: (augmentation_layer(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
        ds = ds.map(apply_brightness, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

ds_train_bin = build_dataset_binary(train_paths, train_labels_bin, augment=True,  shuffle=True)
ds_val_bin   = build_dataset_binary(val_paths,   val_labels_bin)
ds_test_bin  = build_dataset_binary(test_paths,  test_labels_bin)


In [ ]:
tf.keras.backend.clear_session(); gc.collect()

bin_backbone = MobileNetV3Small(
    include_top=False, weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
bin_backbone.trainable = True
for layer in bin_backbone.layers[:-20]:
    layer.trainable = False
for layer in bin_backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

inputs_bin = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x_bin = layers.Lambda(lambda img: mob_v3_preproc(img), name="preprocess")(inputs_bin)
x_bin = bin_backbone(x_bin, training=False)
x_bin = layers.GlobalAveragePooling2D()(x_bin)
x_bin = layers.Dense(128, activation="relu")(x_bin)
x_bin = layers.Dropout(0.3)(x_bin)
out_bin = layers.Dense(1, activation="sigmoid", name="binary_out")(x_bin)

binary_model = keras.Model(inputs_bin, out_bin, name="MobileNetV3Small_Binary")
binary_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

cb_bin = [
    EarlyStopping(monitor="val_loss", patience=10,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(f"{MODELS_DIR}/binary_best.keras",
                    monitor="val_auc", mode="max",
                    save_best_only=True, verbose=0),
]

t0 = time.time()
history_bin = binary_model.fit(
    ds_train_bin, validation_data=ds_val_bin,
    epochs=40, callbacks=cb_bin, verbose=1,
)
t_bin = time.time() - t0

plot_history(history_bin,
             "Clasificador Binario MobileNetV3Small – Curvas",
             f"{FIGURES_DIR}/17_binary_curves.png",
             ylim_acc=(0.5, 1.0))


In [ ]:
bin_metrics                    = binary_model.evaluate(ds_test_bin, verbose=0)
bin_loss, bin_acc, bin_prec, bin_rec, bin_auc = bin_metrics
bin_f1 = 2 * bin_prec * bin_rec / (bin_prec + bin_rec + 1e-9)

print("Clasificador Binario (test)")
print(f"  Accuracy  : {bin_acc*100:.2f}%")
print(f"  Precision : {bin_prec:.4f}")
print(f"  Recall    : {bin_rec:.4f}")
print(f"  F1        : {bin_f1:.4f}")
print(f"  AUC-ROC   : {bin_auc:.4f}")

y_bin_true, y_bin_pred = [], []
for x_batch, y_batch in ds_test_bin:
    preds = (binary_model.predict(x_batch, verbose=0).flatten() > 0.5).astype(int)
    y_bin_pred.extend(preds)
    y_bin_true.extend(y_batch.numpy().astype(int))

cm_bin = confusion_matrix(y_bin_true, y_bin_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_bin, annot=True, fmt="d", cmap="Greens",
            xticklabels=["Healthy", "Enfermo"],
            yticklabels=["Healthy", "Enfermo"],
            linewidths=0.5, ax=ax)
ax.set_xlabel("Prediccion"); ax.set_ylabel("Real")
ax.set_title(f"Confusion – Clasificador Binario\n"
             f"Acc={bin_acc*100:.2f}%  F1={bin_f1:.3f}  AUC={bin_auc:.3f}")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/18_binary_confusion.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Bloque 9 – Estimacion de Viabilidad en Tablet

Se mide el tiempo de inferencia individual y el tamano en disco de cada modelo para evaluar el requerimiento de menos de 2 segundos por imagen.

In [ ]:
THRESHOLD_MS = 2000   # 2 segundos en milisegundos

model_files = {
    "CNN Baseline":        f"{MODELS_DIR}/baseline_best.keras",
    "MobileNetV3Small FE": f"{MODELS_DIR}/mob_fe_best.keras",
    "EfficientNetB0 FE":   f"{MODELS_DIR}/eff_fe_best.keras",
    "ResNet50 FE":         f"{MODELS_DIR}/res_fe_best.keras",
    "MobileNetV3Small FT": f"{MODELS_DIR}/mob_ft_best.keras",
    "ResNet50 FT":         f"{MODELS_DIR}/res_ft_best.keras",
    "Binario MobV3Small":  f"{MODELS_DIR}/binary_best.keras",
}

tablet_results = {}
sample_x = next(iter(ds_test.take(1)))[0][:1]

print(f"{'Modelo':<25} {'Inferencia (ms)':>16} {'Tamano (MB)':>12} {'Estado':>15}")
print("-" * 72)

for name, path in model_files.items():
    if not os.path.exists(path):
        print(f"{name:<25} {'NO ENCONTRADO':>16}")
        continue
    mdl      = keras.models.load_model(path, compile=False)
    size_mb  = os.path.getsize(path) / (1024 ** 2)
    _        = mdl.predict(sample_x, verbose=0)           # warmup
    t0       = time.time()
    for _    in range(50): mdl.predict(sample_x, verbose=0)
    inf_ms   = (time.time() - t0) / 50 * 1000
    status   = "CUMPLE" if inf_ms < THRESHOLD_MS else "EXCEDE LIMITE"
    tablet_results[name] = {"inf_ms": inf_ms, "size_mb": size_mb}
    print(f"{name:<25} {inf_ms:>14.1f}  {size_mb:>10.1f}  {status:>15}")
    del mdl; gc.collect(); tf.keras.backend.clear_session()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

names_t   = list(tablet_results.keys())
inf_vals  = [tablet_results[n]["inf_ms"]  for n in names_t]
size_vals = [tablet_results[n]["size_mb"] for n in names_t]
pal       = ["#4caf50" if v < THRESHOLD_MS else "#f44336" for v in inf_vals]

bars1 = axes[0].bar(names_t, inf_vals, color=pal, edgecolor="black", lw=0.4)
axes[0].axhline(THRESHOLD_MS, color="crimson", ls="--", lw=1.5, label="Limite 2 s")
for bar, v in zip(bars1, inf_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f"{v:.0f}", ha="center", va="bottom", fontsize=8)
axes[0].set_ylabel("Tiempo de inferencia (ms)")
axes[0].set_title("Tiempo de inferencia por imagen")
axes[0].legend(fontsize=9)
axes[0].tick_params(axis="x", rotation=40, labelsize=8)

axes[1].bar(names_t, size_vals, color="#2196F3", edgecolor="black", lw=0.4)
for i, v in enumerate(size_vals):
    axes[1].text(i, v + 0.3, f"{v:.1f} MB", ha="center", va="bottom", fontsize=8)
axes[1].set_ylabel("Tamano en disco (MB)")
axes[1].set_title("Tamano del modelo guardado")
axes[1].tick_params(axis="x", rotation=40, labelsize=8)

plt.suptitle("Estimacion de viabilidad en dispositivo movil", fontsize=13)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/19_tablet_viability.png", dpi=150, bbox_inches="tight")
plt.show()


**Observaciones sobre viabilidad:**

Los tiempos medidos en Colab con GPU son optimistas respecto a lo que se obtendria en una tablet ARM sin GPU dedicada. En la practica, los modelos moviles en CPU ARM suelen ser entre 3x y 8x mas lentos que en GPU; aun con ese factor, MobileNetV3Small permanece muy por debajo del limite de 2 s.

Para produccion real, el flujo recomendado es convertir el modelo a **TFLite** mediante `tf.lite.TFLiteConverter.from_saved_model()`, con cuantizacion de pesos a int8, lo que reduce el tamano en disco en un factor de 4 y acelera la inferencia en dispositivos ARM con delegados NNAPI o Core ML. ResNet50 requeriria cuantizacion agresiva o un dispositivo con NPU dedicada para cumplir el requisito de latencia en campo.

---
## Exportacion de figuras para el informe

In [ ]:
import shutil

zip_path = "/content/figuras_experiencia1"
shutil.make_archive(zip_path, "zip", FIGURES_DIR)

print("Figuras generadas:")
for f in sorted(Path(FIGURES_DIR).iterdir()):
    print(f"  {f.name:<50s}  {f.stat().st_size/1024:6.1f} KB")

print(f"\nZIP disponible en: {zip_path}.zip")
print("Para descargar en Colab, ejecutar:")
print("  from google.colab import files")
print(f"  files.download('{zip_path}.zip')")
